# Interpretability Benchmarks

Standard quantitative metrics for evaluating mechanistic interpretability
experiments. These metrics let you rigorously compare interventions,
measure faithfulness of explanations, and benchmark circuit discovery.

This notebook covers the `irtk.interp_benchmarks` module.

## Why This Matters

Interpretability benchmarks provide standardized metrics for evaluating explanations: logit difference, KL divergence, loss recovered, and ablation effect size. Consistent metrics make it possible to compare findings across studies and validate circuit claims.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import interp_benchmarks

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Logit Difference

The standard metric for IOI and similar tasks: how much does the
model prefer the correct answer over an alternative?

In [ ]:
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)
logits = model(tokens)

paris_id = tokenizer.encode(" Paris")[0]
london_id = tokenizer.encode(" London")[0]
rome_id = tokenizer.encode(" Rome")[0]

ld_paris_london = interp_benchmarks.logit_diff(logits, paris_id, london_id)
ld_paris_rome = interp_benchmarks.logit_diff(logits, paris_id, rome_id)

print(f"Logit diff (Paris vs London): {ld_paris_london:.4f}")
print(f"Logit diff (Paris vs Rome):   {ld_paris_rome:.4f}")
print(f"\nPositive = model prefers the correct answer")

## 2. KL Divergence

Measure how much an intervention changes the output distribution.

In [ ]:
clean_logits = model(tokens)

# Measure KL for progressively stronger interventions
kl_values = []
scales = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
for scale in scales:
    def noise_hook(x, name):
        return x + scale * jnp.ones_like(x)
    
    mod_logits = model.run_with_hooks(
        tokens, fwd_hooks=[("blocks.6.hook_attn_out", noise_hook)]
    )
    kl = interp_benchmarks.kl_divergence(clean_logits, mod_logits)
    kl_values.append(kl)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scales, kl_values, 'bo-')
ax.set_xlabel('Noise scale')
ax.set_ylabel('KL divergence')
ax.set_title('KL Divergence vs Intervention Strength (Layer 6 Attn)')
plt.tight_layout()
plt.show()

## 3. Loss Recovered

What fraction of the original performance is preserved after
an intervention? 1.0 = perfect, 0.0 = as bad as full corruption.

In [ ]:
# Compute loss recovered for ablation at each layer
recoveries = []
for layer in range(model.cfg.n_layers):
    hook = f"blocks.{layer}.hook_attn_out"
    
    def zero_hook(x, name):
        return jnp.zeros_like(x)
    
    mod_logits = model.run_with_hooks(tokens, fwd_hooks=[(hook, zero_hook)])
    rec = interp_benchmarks.loss_recovered(model, tokens, mod_logits)
    recoveries.append(rec)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['green' if r > 0.9 else 'orange' if r > 0.5 else 'red' for r in recoveries]
ax.bar(range(model.cfg.n_layers), recoveries, color=colors)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Loss recovered')
ax.set_title('Loss Recovered After Zero-Ablating Attention (per layer)')
plt.tight_layout()
plt.show()

## 4. Ablation Effect Size

Normalized impact of ablating a hook point. Supports both
zero and mean ablation.

In [ ]:
def my_metric(logits):
    return float(logits[-1, paris_id])

print("Ablation effect sizes (zero ablation):")
print(f"{'Hook':<35s} {'Effect':>8s} {'Size':>8s}")
print('-' * 55)

for layer in range(model.cfg.n_layers):
    for component in ['hook_attn_out', 'hook_mlp_out']:
        hook = f"blocks.{layer}.{component}"
        result = interp_benchmarks.ablation_effect_size(
            model, tokens, hook, my_metric
        )
        if result['effect_size'] > 0.01:
            print(f"  {hook:<33s} {result['effect']:+.4f} {result['effect_size']:.4f}")

In [ ]:
# Compare zero vs mean ablation
zero_effects = []
mean_effects = []
for layer in range(model.cfg.n_layers):
    hook = f"blocks.{layer}.hook_attn_out"
    z = interp_benchmarks.ablation_effect_size(model, tokens, hook, my_metric, "zero")
    m = interp_benchmarks.ablation_effect_size(model, tokens, hook, my_metric, "mean")
    zero_effects.append(z['effect_size'])
    mean_effects.append(m['effect_size'])

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(model.cfg.n_layers)
ax.bar(x - 0.2, zero_effects, 0.4, label='Zero ablation', color='steelblue')
ax.bar(x + 0.2, mean_effects, 0.4, label='Mean ablation', color='coral')
ax.set_xlabel('Layer')
ax.set_ylabel('Effect size')
ax.set_title('Zero vs Mean Ablation Effect Size')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Faithfulness Correlation

Does a set of attribution scores actually predict which hooks
matter when ablated? A faithful attribution method should correlate
with actual ablation effects.

In [ ]:
# Generate attribution scores from direct logit attribution
from irtk.circuits import direct_logit_attribution
_, cache = model.run_with_cache(tokens)
head_attrs = direct_logit_attribution(model, cache, token=paris_id)

# Build attribution dict for attention outputs
attributions = {}
for layer in range(model.cfg.n_layers):
    hook = f"blocks.{layer}.hook_attn_out"
    # Sum absolute head attributions for this layer
    attributions[hook] = float(np.sum(np.abs(head_attrs[layer])))

result = interp_benchmarks.faithfulness_correlation(
    model, tokens, attributions, my_metric
)

print(f"Faithfulness correlation: {result['correlation']:.4f}")
print(f"  (Pearson r between attributions and ablation effects)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(result['attribution_scores'], result['ablation_effects'], color='steelblue', s=50)
for i, name in enumerate(result['hook_names']):
    layer = name.split('.')[1]
    ax.annotate(f'L{layer}', (result['attribution_scores'][i], result['ablation_effects'][i]),
                fontsize=8, ha='center', va='bottom')
ax.set_xlabel('Attribution score')
ax.set_ylabel('Actual ablation effect')
ax.set_title(f'Attribution vs Ablation (r={result["correlation"]:.3f})')
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `logit_diff()` | Difference between correct and incorrect logits |
| `kl_divergence()` | KL(original \|\| modified) distribution divergence |
| `loss_recovered()` | Fraction of original loss preserved after intervention |
| `ablation_effect_size()` | Normalized impact of ablating a hook point |
| `faithfulness_correlation()` | How well attributions predict ablation effects |